In [2]:
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
url = 'https://www.flipkart.com/apple-iphone-18-pro-burgundy-256-gb/p/itm2b2c10b01cc46?pid=MOBHQT5JBPDHYHJM&lid=LSTMOBHQT5JBPDHYHJM3ZGIUY&marketplace=FLIPKART&q=iphone+18+pro&store=tyy%2F4io&srno=s_1_1&otracker=AS_Query_OrganicAutoSuggest_1_8_na_na_na&otracker1=AS_Query_OrganicAutoSuggest_1_8_na_na_na&fm=organic&iid=d3d44326-cdd2-436c-b8b6-861371a00734.MOBHQT5JBPDHYHJM.SEARCH&ppt=browse&ppn=browse&ssid=u0plp33duo0000001791960023948&qH=f969d826bf8e013f&ov_redirect=true'

loader = WebBaseLoader(url)

docs = loader.load()

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

chunks = splitter.split_documents(docs)

len(chunks)

8

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\ManishAgrahari\Desktop\my_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1662.43it/s]


In [10]:
vector = Chroma.from_documents(documents=chunks, embedding=embeddings)
vector

In [9]:
retriever = vector.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000026E645A06E0>, search_kwargs={})

In [23]:
system_data = """
You are an helpful question answering system.
and if you dont know the answer directly say i dont know.
keep the answer concise.
{context}
"""

prompt = ChatPromptTemplate(
    [
        ('system', system_data),
        ('human'), '{input}'
    ]
)

In [24]:
llm = ChatGroq(model='openai/gpt-oss-120b')

In [25]:
ques_ans_chain = create_stuff_documents_chain(llm, prompt)

In [26]:
rag_chain = create_retrieval_chain(retriever, ques_ans_chain)

In [31]:
response = rag_chain.invoke(
    {'input': 'what is the price of i phone 18 pro'}
)
print(response['answer'])

The Apple iPhone 18 Pro (256 GB, Burgundy) is listed for **₹1,64,900** on Flipkart (₹299 Protect Promise fee extra). With available offers, the price can be as low as **₹1,59,150**.


In [32]:
response = rag_chain.invoke(
    {'input': 'in what all colors it is available'}
)
print(response['answer'])

The listing only mentions one color for the iPhone 18 Pro 256 GB – **Burgundy**. No other colors are specified.


In [49]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

In [50]:
context_system = 'Given a chat history and latest user question which might refernce context in chat history'

In [ ]:
context_prompt = ChatPromptTemplate.from_messages(
    [
        ('system', "Rephrame the query as Standalone query based on past context or chat history"),
        MessagesPlaceholder('chat_history'),
        ('human', '{input}')
    ]
)

In [52]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ('system', context_system),
        MessagesPlaceholder('chat_history'),
        ('human', '{input} \n\n Context: \n {context}')
    ]
)

In [53]:
history_aware_retriever = create_history_aware_retriever(llm, retriever, context_prompt)

In [54]:
ques_ans_chain = create_stuff_documents_chain(llm, qa_prompt)

In [55]:
rag_chain = create_retrieval_chain(history_aware_retriever, ques_ans_chain)

In [56]:
chat_history = []

In [57]:
question = 'what is the price of i phone 18 pro'

In [58]:
response = rag_chain.invoke(
    {'input': question, 'chat_history': chat_history}
)

In [59]:
chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response['answer'])
    ]
)

In [60]:
print(response['answer'])

The iPhone 18 Pro is listed on Flipkart at the following prices (Indian rupees):

| Storage | List price | Discounted price (when offers are applied) |
|---------|------------|--------------------------------------------|
| 256 GB | **₹1,64,900** | ~₹1,59,150 |
| 512 GB | **₹1,89,900** | – |
| 1 TB   | **₹2,39,900** | – |
| 2 TB   | **₹3,14,900** | – |

So the base (non‑discounted) price for the 256 GB model – the entry‑level iPhone 18 Pro – is **₹1,64,900**. Prices can vary slightly depending on the seller, ongoing promotions, or exchange offers.


In [61]:
question2 = 'About what i have asked about I phone 18 pro'

In [62]:
response2 = rag_chain.invoke(
    {'input': question2, 'chat_history': chat_history}
)

In [63]:
print(response2['answer'])

Here’s a quick summary of the **Apple iPhone 18 Pro (Burgundy, 256 GB)** listing you pasted from Flipkart:

| Detail | Information |
|--------|--------------|
| **Base price (list price)** | **₹1,64,900** |
| **Current “Buy at” price with offers** | **₹1,59,150** (plus a ₹299 Protect‑Promise fee if you opt for the protection plan) |
| **Storage variants & their list prices** | 256 GB – ₹1,64,900  <br>512 GB – ₹1,89,900  <br>1 TB – ₹2,39,900  <br>2 TB – ₹3,14,900 |
| **Ratings** | 4.6 ★ from 281 reviews |
| **Seller** | TREASUREHAUL ONLINE (6‑year Flipkart seller) |
| **Warranty** | Apple One‑year limited warranty + 7‑day return policy |
| **Delivery** | Expected by **Saturday, 17 Oct** (once you set a delivery pin) |
| **EMI options** | Example: **₹27,819 × 6 months** (total ₹1,66,916) |
| **Exchange offers** | Up to **₹71,650** off (depends on your current device & pincode) |
| **Bank & coupon discounts** | • Up to **₹5,750** off with select bank cards  <br>• Apple coupon: **₹1,000** 

In [64]:
chat_history

[HumanMessage(content='what is the price of i phone 18 pro', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The iPhone\u202f18\u202fPro is listed on Flipkart at the following prices (Indian rupees):\n\n| Storage | List price | Discounted price (when offers are applied) |\n|---------|------------|--------------------------------------------|\n| 256\u202fGB | **₹1,64,900** | ~₹1,59,150 |\n| 512\u202fGB | **₹1,89,900** | – |\n| 1\u202fTB   | **₹2,39,900** | – |\n| 2\u202fTB   | **₹3,14,900** | – |\n\nSo the base (non‑discounted) price for the 256\u202fGB model – the entry‑level iPhone\u202f18\u202fPro – is **₹1,64,900**. Prices can vary slightly depending on the seller, ongoing promotions, or exchange offers.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]